# QLoRA training on Kaggle (T4)

Thin wrapper: all logic lives in the `legal_ft` package (`src/legal_ft/train.py`), the same code path the local dry run (`scripts/dry_run_train.py`) exercises.

**Settings:** Accelerator *GPU T4 x1*, Internet *on*. Optional secret `HF_TOKEN` (faster model download; needed later to push the adapter).

**Run it as a background job:** *Save Version → Save & Run All (Commit)*. Expect several hours for 4,500 examples (≈281 optimizer steps at effective batch 16); `[progress]` lines report seconds per step and an ETA from the first minute on.

**First time:** run the cells interactively until the first `[progress] step 1` / `step 2` lines appear, check the s/step and ETA fit Kaggle's 12 h limit, then stop and commit. Outputs land in `/kaggle/working/qlora/`.

In [ ]:
# Use exactly one GPU. Must run before torch initializes CUDA. On "T4 x2" the Trainer
# would otherwise try multi-GPU code paths that a 4-bit model pinned to one card cannot use.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Clone + editable install: configs/ is read relative to the source tree, so a plain
# `pip install git+...` would not find it.
!git clone -q https://github.com/raghavpli515/Legal-contract-LLM-fine-tuning-with-evaluation.git /tmp/repo
%cd /tmp/repo
!git log -1 --format='commit %h %s'
!pip install -q -e ".[train]"

# An editable install registers itself via a .pth file, which Python only reads at startup;
# this kernel is already running, so put the source tree on the path directly.
import importlib, sys
sys.path.insert(0, "/tmp/repo/src")
importlib.invalidate_caches()

import legal_ft, transformers, trl
print("python", sys.version.split()[0], "| legal_ft from", legal_ft.__file__)
assert int(transformers.__version__.split(".")[0]) >= 5, (
    f"transformers {transformers.__version__} is too old: the pip install above failed, check its output")

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret (fine for training):", type(e).__name__)

import torch, transformers, trl, peft, bitsandbytes
print(torch.cuda.get_device_name(0), "| visible GPUs:", torch.cuda.device_count())
assert torch.cuda.device_count() == 1
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__,
      "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)

## Build the data
Deterministic (~20 s): downloads CUAD (sha256-checked) and rebuilds the JSONL files. The checksums must match `configs/data.yaml` so this run uses byte-identical data to the local build.

In [ ]:
from legal_ft.data.build_sft import build
stats = build()
assert not stats["sha256_mismatch"], f"data differs from reference build: {stats['sha256_mismatch']}"
print({k: v["n"] for k, v in stats["files"].items()})

## Train
`train()` checks every example fits `max_length` (no truncation) and that loss is on the answer tokens only, then trains and saves the adapter + `log_history.json`.

In [ ]:
from pathlib import Path
from legal_ft.config import REPO_ROOT, load_config
from legal_ft.train import train

cfg = load_config("train_qlora")
data = REPO_ROOT / "data" / "processed"
out = Path("/kaggle/working/qlora")
adapter = train(cfg, cfg["model_id"], data / "train.jsonl", data / "val.jsonl", out)
print("adapter saved to", adapter)

In [ ]:
import json
h = json.loads((out / "log_history.json").read_text())
print("train loss:", [round(x["loss"], 3) for x in h if "loss" in x][::5])
print("eval loss: ", [round(x["eval_loss"], 3) for x in h if "eval_loss" in x])
print("peak GPU GB:", round(torch.cuda.max_memory_reserved() / 1024**3, 2))
!du -sh {out}/adapter

## Push adapter to the Hub
Milestone 2: after the eval confirms the adapter is worth publishing.